In [21]:
from ollama import chat

## **Model**

In [22]:
MODEL = "deepseek-r1:1.5b"
MAX_STEPS = 5

## **Tool**

Composed of three parts: 
- doc string
- type hints eg "city"
- name

In [23]:
def get_weather(city:str) -> str:
    """ Get current weather for a given city"""
    return f"22°C and light rain experienced in {city} "

In [24]:
TOOLS = {
    "get_weather":get_weather,
}

In [25]:
TOOLS.keys()

dict_keys(['get_weather'])

In [26]:
TOOLS.values()

dict_values([<function get_weather at 0x7f332a405b20>])

In [27]:
print(TOOLS)

{'get_weather': <function get_weather at 0x7f332a405b20>}


## **Agent Loop**

COmposed of loop with maximum nuumber of steps, this acts as a guardrail to limit the number of calls made by the model

In [28]:
def run_agent(user_question: str, max_steps=MAX_STEPS):
    # The conversation history. The model has no memory,
    # so we send the whole history on every call.
    messages = [
        {"role": "system", "content": "You can use tools. After you get a tool result, use it to answer the user."},
        {"role": "user", "content": user_question},
    ]

    for step in range(max_steps):
        # 1. Ask the model what to do next
        response = chat(
            model=MODEL,
            messages=messages,
            tools= list(TOOLS.values()),
        )

        # 2. Save the model's reply in the history
        messages.append(response.message)

        # 3. No tool requested? Then this is the final answer -> stop
        if not response.message.tool_calls:
            return response.message.content

        # 4. The model asked for tool(s): run each one
        for call in response.message.tool_calls:
            name = call.function.name          # e.g. "get_weather"
            args = call.function.arguments    # e.g. {"city": "Nairobi"}

            # Small models sometimes wrap the real arguments inside "parameters",
            # like a box inside a box. Keep opening until we reach the real ones.
            while "parameters" in args:
                args = args["parameters"]

            print(f"step {step + 1}, model requested: {name}{args}")

            try:
                result = TOOLS[name](**args)   # run the real Python function
            except Exception as e:
                result = f"ERROR: {e}"         # send errors back instead of crashing

            print(f"   tool returned: {result}")

            # 5. Give the tool's result back to the model
            messages.append({"role": "tool", "content": str(result), "tool_name": name})

    return f"Stopped: reached maximum number of steps ({max_steps})"


In [29]:
   # Needs a tool -> watch the loop run.
print(run_agent("Should I carry an umbrella in Nairobi today?"))
print("-" * 60)



In Nairobi, the best decision to carry an umbrella depends on the weather forecast and personal considerations:

1. **Weather Forecast**: Check the weather forecast for Nairobi today. If there's a chance of rain or partial sun, carrying an umbrella would be advisable.

2. **Umbrella Availability**: Assess how many umbrellas you have on hand. If you have extra umbrellas, you can carry them safely, especially if it's a busy area.

3. **Number of Umbrellas Needed**: For a quiet, safe spot, carry fewer umbrellas. For a busy area, maybe carry more.

4. **Clothing Carry**: If you decide to take extra clothes, consider whether it's necessary or if you can manage without them, potentially saving space.

5. **Personal Comfort**: Reflect on whether an umbrella is too much. If not, carry just a few extra clothes for last-resort situations.

By evaluating these factors, you can make an informed decision about whether to carry an umbrella in Nairobi today.
----------------------------------------